## IBGE

SRAG Bronze -> Silver, 2020–2024. Executar como script ou com %run no Jupyter.

Dependências: pandas, pyarrow. Não conecta ao PostgreSQL.
Mantém todas as linhas e as 194 colunas; não deduplica nem filtra COVID.
Padroniza quatro códigos (sem decodificar significados clínicos), preservando
seus valores originais. Demais campos conservam seus tipos Arrow de origem.
Datas que ainda são texto na Bronze continuam texto nesta primeira versão.
Ausência de marcador laboratorial nunca é convertida para negativo.

Chave proposta: (ano_arquivo, nu_notific), validada em toda a execução.
Esta chave identifica notificações da extração, não pessoas/casos únicos.
Ano de análise: ano de DT_SIN_PRI, sem fallback para nome do arquivo.
Vínculos municipais por papel e ano de análise; não vincula estabelecimentos
ao CNES por nome. Não cria foreign keys: pendências precisam ser avaliadas.

In [15]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv
import hashlib
import json
import sqlite3
import uuid
import unicodedata

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [16]:
PROJETO = Path(r"D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil")
BRONZE = PROJETO / "Camada_Bronze" / "dados_srag"
SILVER = PROJETO / "Camada_Silver"
# Referência explicitamente validada no projeto. Ajuste se mover o arquivo.
IBGE = SILVER / "dados/populacao_ibge/20261024_163052_e1db3d2b/populacao_municipios_2020_2024.parquet"
ANOS = tuple(range(2020, 2025))
LOTE = 50_000
CODIGOS = ("CLASSI_FIN", "EVOLUCAO", "PCR_SARS2", "AN_SARS2")
PAPEIS = {
    "notificacao": ("CO_MUN_NOT", "SG_UF_NOT"),
    "residencia": ("CO_MUN_RES", "SG_UF"),
    "internacao": ("CO_MU_INTE", "SG_UF_INTE"),
}

In [17]:
# Equivalências explícitas do diagnóstico deste projeto, versão 2026-10-28.
# Agregação do DF ao nível municipal IBGE; não é população de cada RA.
CODIGOS_LOCAIS_DF = {
    "530005", "530015", "530020", "530030", "530040", "530050",
    "530060", "530070", "530080", "530090", "530100", "530110",
    "530120", "530130", "530135", "530140", "530150", "530160",
    "530170", "530180", "539901", "539913", "539917", "539922",
    "539925", "539928", "539929", "539930", "539931", "539933", "539934",
}
CAMPOS_NOME = {
    "notificacao": "ID_MUNICIP",
    "residencia": "ID_MN_RESI",
    "internacao": "ID_MN_INTE",
}


def texto(s):
    return s.astype("string").str.strip().replace("", pd.NA)


def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()


def referencia_ibge(path):
    cols = ["ano_analise", "codigo_ibge_7", "uf", "municipio", "populacao"]
    ref = pd.read_parquet(path, columns=cols)
    ref["ano_analise"] = pd.to_numeric(ref["ano_analise"], errors="raise").astype("Int64")
    ref = ref.loc[ref.ano_analise.isin(ANOS)].copy()
    for c in ["codigo_ibge_7", "uf", "municipio"]:
        ref[c] = texto(ref[c])
    ref["uf"] = ref.uf.str.upper()
    if set(ref.ano_analise) != set(ANOS):
        raise ValueError("IBGE não cobre todos os anos solicitados.")
    if not ref.codigo_ibge_7.str.fullmatch(r"[0-9]{7}", na=False).all():
        raise ValueError("Código IBGE de referência inválido.")
    if ref[["uf", "municipio"]].isna().any().any():
        raise ValueError("IBGE contém UF ou nome municipal ausente.")
    ref["codigo_6"] = ref.codigo_ibge_7.str[:6]
    if ref.duplicated(["ano_analise", "codigo_6"]).any():
        raise ValueError("Referência ambígua: código municipal de 6 dígitos/ano repetido.")
    return ref.rename(columns={"uf": "uf_ibge", "municipio": "municipio_ibge"})


def vincular(df, ref, papel, codigo, uf):
    base = pd.DataFrame({
        "ano_analise": df["ano_analise"],
        "codigo_original": texto(df[codigo]),
        "uf_original": texto(df[uf]).str.upper(),
    })
    base["codigo_6"] = base["codigo_original"]
    base["metodo"] = "CODIGO_DIRETO"
    eq_df = (base.uf_original.eq("DF") & base.codigo_original.isin(CODIGOS_LOCAIS_DF)).fillna(False)
    base.loc[eq_df, "codigo_6"] = "530010"
    base.loc[eq_df, "metodo"] = "AGREGACAO_DF"
    campo_nome = CAMPOS_NOME[papel]
    if campo_nome in df.columns:
        nomes = texto(df[campo_nome]).map(
            lambda x: "".join(c for c in unicodedata.normalize("NFKD", str(x).upper())
                              if not unicodedata.combining(c)) if pd.notna(x) else ""
        ).str.replace(r"\s+", " ", regex=True).str.strip()
        eq_rs = (base.uf_original.eq("RS") & base.codigo_original.eq("431453")
                 & nomes.eq("PINTO BANDEIRA")).fillna(False)
        base.loc[eq_rs, "codigo_6"] = "431454"
        base.loc[eq_rs, "metodo"] = "EQUIVALENCIA_PINTO_BANDEIRA"
    m = base.merge(ref, on=["ano_analise", "codigo_6"], how="left", validate="many_to_one", sort=False)
    # merge preserva a ordem esquerda; índice do lote sempre é reiniciado.
    status = pd.Series("SEM_CORRESPONDENCIA", index=m.index, dtype="string")
    formato = m.codigo_original.str.fullmatch(r"[0-9]{6}", na=False)
    encontrado = m.codigo_ibge_7.notna()
    conflito = encontrado & m.uf_original.notna() & m.uf_original.ne(m.uf_ibge)
    status.loc[encontrado] = "VINCULADO_COM_POPULACAO"
    status.loc[encontrado & m.metodo.eq("AGREGACAO_DF")] = "VINCULADO_AGREGACAO_DF_COM_POPULACAO"
    status.loc[encontrado & m.metodo.eq("EQUIVALENCIA_PINTO_BANDEIRA")] = "VINCULADO_EQUIVALENCIA_RS_COM_POPULACAO"
    status.loc[encontrado & m.populacao.isna()] = "VINCULADO_SEM_POPULACAO"
    status.loc[conflito] = "UF_DIVERGENTE"
    status.loc[~formato] = "CODIGO_INVALIDO"
    status.loc[m.codigo_original.isna()] = "CODIGO_AUSENTE"
    status.loc[~m.ano_analise.isin(ANOS)] = "ANO_FORA_REFERENCIA"
    status.loc[m.ano_analise.isna()] = "ANO_AUSENTE"
    valido = status.isin(["VINCULADO_COM_POPULACAO", "VINCULADO_SEM_POPULACAO",
                         "VINCULADO_AGREGACAO_DF_COM_POPULACAO",
                         "VINCULADO_EQUIVALENCIA_RS_COM_POPULACAO"])
    df[f"codigo_ibge_7_{papel}"] = m.codigo_ibge_7.where(valido).astype("string")
    df[f"municipio_ibge_{papel}"] = m.municipio_ibge.where(valido).astype("string")
    df[f"uf_ibge_{papel}"] = m.uf_ibge.where(valido).astype("string")
    df[f"status_vinculo_{papel}"] = status
    df[f"fl_uf_ausente_{papel}"] = m.uf_original.isna()
    return status


def tipo_sql(tipo):
    if pa.types.is_string(tipo) or pa.types.is_large_string(tipo):
        return "TEXT"
    if pa.types.is_boolean(tipo):
        return "BOOLEAN"
    if pa.types.is_integer(tipo):
        return "BIGINT"
    if pa.types.is_floating(tipo):
        return "DOUBLE PRECISION"
    if pa.types.is_decimal(tipo):
        return f"NUMERIC({tipo.precision}, {tipo.scale})"
    if pa.types.is_timestamp(tipo):
        return "TIMESTAMPTZ" if tipo.tz else "TIMESTAMP"
    if pa.types.is_date(tipo):
        return "DATE"
    raise ValueError(f"Tipo SQL não previsto: {tipo}")


def main():
    print("Iniciando tratamento SRAG — conferindo arquivos e referência IBGE...", flush=True)
    fontes = [BRONZE / f"srag_{ano}.parquet" for ano in ANOS]
    for path in [*fontes, IBGE]:
        if not path.is_file():
            raise FileNotFoundError(path)
    ref = referencia_ibge(IBGE)
    esquemas = [pq.read_schema(p) for p in fontes]
    primeiro = esquemas[0]
    if len({n.lower() for n in primeiro.names}) != len(primeiro.names):
        raise ValueError("Colunas colidem após conversão para minúsculas.")
    obrigatorias = {"NU_NOTIFIC", "DT_SIN_PRI", "DT_NOTIFIC", *CODIGOS}
    for codigo, uf in PAPEIS.values():
        obrigatorias.update([codigo, uf])
    for esquema in esquemas:
        if set(esquema.names) != set(primeiro.names) or not obrigatorias.issubset(esquema.names):
            raise ValueError("Colunas mudaram entre arquivos ou faltam campos obrigatórios.")
        for campo in primeiro:
            if campo.name not in CODIGOS and esquema.field(campo.name).type != campo.type:
                raise ValueError(f"Tipo divergente não previsto: {campo.name}")

    agora = datetime.now(timezone.utc)
    run = agora.strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
    dados = SILVER / "dados/srag" / run
    qualidade = SILVER / "qualidade/srag" / run
    doc = SILVER / "documentacao/srag" / run
    for p in [dados, qualidade, doc]:
        p.mkdir(parents=True, exist_ok=False)

    extras = [pa.field(c.lower() + "_original", primeiro.field(c).type) for c in CODIGOS]
    extras += [pa.field("ano_arquivo", pa.int16()), pa.field("ano_analise", pa.int16()),
               pa.field("ano_notificacao", pa.int16()), pa.field("linha_arquivo", pa.int64()),
               pa.field("fl_ano_sintomas_diferente_arquivo", pa.bool_()),
               pa.field("fl_notificacao_antes_sintomas", pa.bool_()),
               pa.field("arquivo_origem", pa.string()), pa.field("tratado_em_utc", pa.string())]
    for papel in PAPEIS:
        extras.extend(pa.field(f"{c}_{papel}", pa.string()) for c in
                      ["codigo_ibge_7", "municipio_ibge", "uf_ibge", "status_vinculo"])
        extras.append(pa.field(f"fl_uf_ausente_{papel}", pa.bool_()))
    # Os quatro originais também precisam de tipo uniforme entre anos.
    extras = [pa.field(f.name, pa.string()) if f.name.endswith("_original") else f for f in extras]
    schema = pa.schema([
        pa.field(f.name.lower(), pa.string() if f.name in CODIGOS else f.type)
        for f in primeiro
    ] + extras)
    if len(set(schema.names)) != len(schema.names):
        raise ValueError("Coluna de auditoria colide com coluna de origem.")

    rel = {"status": "EM_PROCESSAMENTO", "execucao": run, "registros": 0,
           "por_ano_arquivo": {}, "avisos": Counter(), "vinculos": {},
           "referencia_ibge": str(IBGE), "sha256_ibge": sha256(IBGE),
           "entradas": [], "chave_proposta": ["ano_arquivo", "nu_notific"],
           "regra_ano_analise": "Ano de DT_SIN_PRI, sem fallback",
           "versao_equivalencias": "2026-10-28.v1",
           "equivalencias": {
               "DF": {"codigos_origem": sorted(CODIGOS_LOCAIS_DF), "destino_ibge_7": "5300108",
                      "regra": "Exige UF DF. Agregação municipal, não população de região administrativa."},
               "RS": {"codigo_origem": "431453", "destino_ibge_7": "4314548",
                      "regra": "Exige UF RS e nome PINTO BANDEIRA normalizado."},
               "999999": "Sem equivalência: preservado sem vínculo",
           },
           "observacao": "Linhas preservadas. Não equivale a pessoas ou casos únicos; não filtra COVID. Datas originalmente texto não foram interpretadas."}
    audit = sqlite3.connect(qualidade / "auditoria_chaves.sqlite")
    audit.execute("CREATE TABLE chaves (ano INTEGER, numero TEXT, n INTEGER, PRIMARY KEY(ano,numero))")
    audit.execute("CREATE TABLE municipios (ano INTEGER, papel TEXT, codigo TEXT, uf TEXT, status TEXT, n INTEGER, PRIMARY KEY(ano,papel,codigo,uf,status))")
    escritor = None
    destino = dados / "srag_2020_2024.parquet"
    parcial = destino.with_suffix(".parquet.part")
    try:
        escritor = pq.ParquetWriter(parcial, schema, compression="snappy")
        for ano, path in zip(ANOS, fontes):
            print(f"Tratando {ano}: {path.name}", flush=True)
            pf = pq.ParquetFile(path)
            rel["entradas"].append({"arquivo": str(path), "sha256": sha256(path), "registros": pf.metadata.num_rows})
            offset = 0
            for batch in pf.iter_batches(batch_size=LOTE):
                df = batch.to_pandas().reset_index(drop=True)
                for c in CODIGOS:
                    original = df[c].astype("string")
                    df[c.lower() + "_original"] = original
                    normal = texto(df[c]).str.replace(r"^([0-9]+)\.0+$", r"\1", regex=True)
                    invalidos = normal.notna() & ~normal.str.fullmatch(r"[0-9]+", na=False)
                    rel["avisos"][f"formato_codigo_invalido_{c}"] += int(invalidos.sum())
                    df[c] = normal  # Valores inesperados são preservados e sinalizados.
                df["ano_arquivo"] = ano
                df["ano_analise"] = df.DT_SIN_PRI.dt.year.astype("Int16")
                df["ano_notificacao"] = df.DT_NOTIFIC.dt.year.astype("Int16")
                df["linha_arquivo"] = range(offset + 1, offset + len(df) + 1)
                df["fl_ano_sintomas_diferente_arquivo"] = df.ano_analise.ne(ano)
                df["fl_notificacao_antes_sintomas"] = (df.DT_NOTIFIC < df.DT_SIN_PRI).astype("boolean")
                df.loc[df.DT_NOTIFIC.isna() | df.DT_SIN_PRI.isna(), "fl_notificacao_antes_sintomas"] = pd.NA
                for c in ["fl_ano_sintomas_diferente_arquivo", "fl_notificacao_antes_sintomas"]:
                    rel["avisos"][c] += int(df[c].sum())
                rel["avisos"]["ano_analise_fora_referencia"] += int((~df.ano_analise.isin(ANOS)).sum())
                numeros = texto(df.NU_NOTIFIC)
                rel["avisos"]["numero_notificacao_ausente"] += int(numeros.isna().sum())
                contagens = numeros.dropna().value_counts()
                audit.executemany("INSERT INTO chaves VALUES(?,?,?) ON CONFLICT(ano,numero) DO UPDATE SET n=n+excluded.n",
                                  [(ano, str(k), int(v)) for k, v in contagens.items()])
                # Armazena o identificador normalizado; Bronze conserva espaços originais.
                df["NU_NOTIFIC"] = numeros
                for papel, (codigo, uf) in PAPEIS.items():
                    status = vincular(df, ref, papel, codigo, uf)
                    acumulado = rel["vinculos"].setdefault(papel, Counter())
                    acumulado.update(status.value_counts().to_dict())
                    rel["avisos"][f"uf_ausente_{papel}"] += int(df[f"fl_uf_ausente_{papel}"].sum())
                    grp = pd.DataFrame({"ano": df.ano_analise.fillna(-1),
                                        "codigo": texto(df[codigo]).fillna("<AUSENTE>"),
                                        "uf": texto(df[uf]).fillna("<AUSENTE>"), "status": status})
                    counts = grp.groupby(["ano", "codigo", "uf", "status"], dropna=False).size()
                    audit.executemany("INSERT INTO municipios VALUES(?,?,?,?,?,?) ON CONFLICT(ano,papel,codigo,uf,status) DO UPDATE SET n=n+excluded.n",
                                      [(int(a), papel, c, u, s, int(n)) for (a,c,u,s), n in counts.items()])
                df["arquivo_origem"] = str(path)
                df["tratado_em_utc"] = agora.isoformat()
                df.columns = df.columns.str.lower()
                escritor.write_table(pa.Table.from_pandas(df[schema.names], schema=schema, preserve_index=False, safe=True))
                offset += len(df)
                rel["registros"] += len(df)
                if offset % 500_000 == 0:
                    print(f"  Processados: {offset:,} / {pf.metadata.num_rows:,}", flush=True)
            if offset != pf.metadata.num_rows:
                raise RuntimeError("Contagem do arquivo divergiu durante a leitura.")
            rel["por_ano_arquivo"][str(ano)] = offset
            audit.commit()
            print(f"  {offset:,} registros preservados", flush=True)
        escritor.close()
        escritor = None
        if pq.ParquetFile(parcial).metadata.num_rows != rel["registros"]:
            raise RuntimeError("Contagem do Parquet final divergente.")
        rel["chaves_duplicadas"] = audit.execute("SELECT COUNT(*) FROM chaves WHERE n>1").fetchone()[0]
        rel["numeros_presentes_em_mais_de_um_arquivo"] = audit.execute("SELECT COUNT(*) FROM (SELECT numero FROM chaves GROUP BY numero HAVING COUNT(*)>1)").fetchone()[0]
        consultas = {
            "chaves_duplicadas.csv": "SELECT * FROM chaves WHERE n>1 ORDER BY ano,numero",
            "vinculos_municipais.csv": "SELECT * FROM municipios ORDER BY ano,papel,codigo",
            "pendencias_municipais.csv": "SELECT * FROM municipios WHERE status <> 'VINCULADO_COM_POPULACAO' ORDER BY ano,papel,codigo",
        }
        for nome, query in consultas.items():
            cursor = audit.execute(query)
            with (qualidade / nome).open("w", encoding="utf-8-sig", newline="") as f:
                w = csv.writer(f, delimiter=";")
                w.writerow([d[0] for d in cursor.description])
                w.writerows(cursor)
        pendencias = any(s != "VINCULADO_COM_POPULACAO" and n for v in rel["vinculos"].values() for s,n in v.items())
        revisar = (rel["chaves_duplicadas"] > 0 or pendencias
                   or rel["avisos"]["numero_notificacao_ausente"] > 0
                   or any(n for k,n in rel["avisos"].items() if k.startswith("formato_codigo_invalido")))
        rel["status"] = "REVISAR_ANTES_DA_CARGA" if revisar else ("VALIDADO_COM_RESSALVAS" if any(rel["avisos"].values()) else "VALIDADO")
        destino_tmp = str(destino)
        rel["parquet_silver"] = destino_tmp
        # DDL é para tabela NOVA, nunca migração nem carga automática.
        colunas = [f'    "{f.name}" {tipo_sql(f.type)}' for f in schema]
        colunas.append('    PRIMARY KEY ("ano_arquivo", "nu_notific")')
        sql = ("-- Revisar relatório de qualidade antes da carga. Não altera tabelas existentes.\n"
               "-- PK identifica notificações por arquivo anual; não identifica pessoas.\n"
               "CREATE SCHEMA IF NOT EXISTS silver;\nCREATE TABLE silver.srag (\n"
               + ",\n".join(colunas) + "\n);\n")
        (doc / "criar_tabela_srag.sql").write_text(sql, encoding="utf-8")
        parcial.rename(destino)
    except Exception as e:
        rel["status"] = "ERRO"
        rel["erro"] = str(e)
        raise
    finally:
        if escritor is not None:
            escritor.close()
        audit.close()
        (qualidade / "relatorio_qualidade.json").write_text(json.dumps(rel, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nStatus: {rel['status']}\nDados: {destino}\nQualidade: {qualidade}\nSQL: {doc / 'criar_tabela_srag.sql'}")
    print("Vínculos:", json.dumps(rel["vinculos"], ensure_ascii=False))
    print("Nenhuma carga foi realizada no PostgreSQL.")
    return rel


if __name__ == "__main__":
    resultado_srag = main()


Iniciando tratamento SRAG — conferindo arquivos e referência IBGE...
Tratando 2020: srag_2020.parquet
  Processados: 500,000 / 1,206,920
  Processados: 1,000,000 / 1,206,920
  1,206,920 registros preservados
Tratando 2021: srag_2021.parquet
  Processados: 500,000 / 1,745,672
  Processados: 1,000,000 / 1,745,672
  Processados: 1,500,000 / 1,745,672
  1,745,672 registros preservados
Tratando 2022: srag_2022.parquet
  Processados: 500,000 / 560,577
  560,577 registros preservados
Tratando 2023: srag_2023.parquet
  279,453 registros preservados
Tratando 2024: srag_2024.parquet
  267,986 registros preservados

Status: REVISAR_ANTES_DA_CARGA
Dados: D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\Camada_Silver\dados\srag\20261028_223707_06cbad50\srag_2020_2024.parquet
Qualidade: D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\Camada_Silver\qualidade\srag\20261028_223707_06cbad50
SQL: D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\Camada_Silver\documentacao\srag\20261028_223

## PostgreSQL

In [18]:
from pathlib import Path
from collections import Counter
from getpass import getpass
import json

import pyarrow.parquet as pq
import psycopg
from psycopg import sql

In [20]:
# CONFIGURAÇÕES
PROJETO = Path(
    r"D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil"
)

EXECUCAO = "20261028_223707_06cbad50"

ARQUIVO_SRAG = (
    PROJETO / "Camada_Silver" / "dados" / "srag"
    / EXECUCAO / "srag_2020_2024.parquet"
)

ARQUIVO_QUALIDADE = (
    PROJETO / "Camada_Silver" / "qualidade" / "srag"
    / EXECUCAO / "relatorio_qualidade.json"
)

CONEXAO = {
    "host": "localhost",
    "port": 5432,
    "dbname": "observatorio_covid19_brasil",
    "user": "postgres",
    "connect_timeout": 15,
}

LOTE = 10_000
PAPEIS = ("notificacao", "residencia", "internacao")


def carregar_srag():
    for arquivo in (ARQUIVO_SRAG, ARQUIVO_QUALIDADE):
        if not arquivo.is_file():
            raise FileNotFoundError(arquivo)

    relatorio = json.loads(
        ARQUIVO_QUALIDADE.read_text(encoding="utf-8")
    )

    # Revisão específica desta execução; não libera qualquer relatório.
    vinculos_esperados = {
        "notificacao": {
            "VINCULADO_COM_POPULACAO": 4049922,
            "CODIGO_AUSENTE": 10542,
            "ANO_FORA_REFERENCIA": 144,
        },
        "residencia": {
            "VINCULADO_COM_POPULACAO": 3973099,
            "VINCULADO_AGREGACAO_DF_COM_POPULACAO": 86714,
            "VINCULADO_EQUIVALENCIA_RS_COM_POPULACAO": 4,
            "CODIGO_AUSENTE": 645,
            "ANO_FORA_REFERENCIA": 144,
            "SEM_CORRESPONDENCIA": 2,
        },
        "internacao": {
            "VINCULADO_COM_POPULACAO": 3596506,
            "CODIGO_AUSENTE": 463958,
            "ANO_FORA_REFERENCIA": 144,
        },
    }

    if relatorio.get("execucao") != EXECUCAO:
        raise ValueError("O relatório pertence a outra execução.")

    if relatorio.get("status") not in {
        "REVISAR_ANTES_DA_CARGA",
        "VALIDADO_COM_RESSALVAS",
        "VALIDADO",
    }:
        raise ValueError("A execução não terminou corretamente.")

    if relatorio.get("chaves_duplicadas") != 0:
        raise ValueError("Há chaves duplicadas ou falta essa validação.")

    avisos = relatorio.get("avisos", {})

    if avisos.get("numero_notificacao_ausente") != 0:
        raise ValueError("Existem notificações sem identificador.")

    if any(
        quantidade
        for nome, quantidade in avisos.items()
        if nome.startswith("formato_codigo_invalido_")
    ):
        raise ValueError("Existem códigos com formato inválido.")

    if relatorio.get("vinculos") != vinculos_esperados:
        raise ValueError(
            "Os vínculos diferem do resultado revisado. "
            "Confira o relatório antes de carregar."
        )

    parquet = pq.ParquetFile(ARQUIVO_SRAG)
    total_esperado = parquet.metadata.num_rows
    colunas = parquet.schema_arrow.names

    if (
        total_esperado != 4_060_608
        or relatorio.get("registros") != total_esperado
    ):
        raise ValueError("Contagem do Parquet ou relatório divergente.")

    senha = getpass("Senha do PostgreSQL: ")

    tabela = sql.Identifier("silver", "srag")
    contagens_arquivo = {papel: Counter() for papel in PAPEIS}

    # O contexto confirma ao finalizar; qualquer erro provoca rollback.
    with psycopg.connect(**CONEXAO, password=senha) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT to_regclass('silver.srag')")
            if cur.fetchone()[0] is None:
                raise RuntimeError(
                    "Crie primeiro silver.srag usando o SQL gerado."
                )

            # Impede outra carga de alterar a tabela durante esta operação.
            cur.execute(
                sql.SQL(
                    "LOCK TABLE {} IN SHARE ROW EXCLUSIVE MODE"
                ).format(tabela)
            )

            cur.execute(
                """
                SELECT column_name
                FROM information_schema.columns
                WHERE table_schema = 'silver'
                  AND table_name = 'srag'
                ORDER BY ordinal_position
                """
            )
            colunas_banco = [linha[0] for linha in cur.fetchall()]

            if set(colunas_banco) != set(colunas):
                raise RuntimeError(
                    "As colunas da tabela diferem das do Parquet.\n"
                    f"Faltam no banco: {set(colunas) - set(colunas_banco)}\n"
                    f"Extras no banco: {set(colunas_banco) - set(colunas)}"
                )

            cur.execute(
                sql.SQL("SELECT EXISTS (SELECT 1 FROM {})").format(tabela)
            )
            if cur.fetchone()[0]:
                raise RuntimeError(
                    "silver.srag já contém dados. "
                    "A carga foi interrompida para evitar duplicação."
                )

            comando_copy = sql.SQL(
                "COPY {} ({}) FROM STDIN"
            ).format(
                tabela,
                sql.SQL(", ").join(map(sql.Identifier, colunas)),
            )

            enviados = 0
            print(
                f"Carregando {total_esperado:,} registros...",
                flush=True,
            )

            with cur.copy(comando_copy) as copy:
                for lote in parquet.iter_batches(batch_size=LOTE):
                    for registro in lote.to_pylist():
                        copy.write_row(
                            tuple(registro[coluna] for coluna in colunas)
                        )

                        for papel in PAPEIS:
                            status = registro[f"status_vinculo_{papel}"]
                            contagens_arquivo[papel][status] += 1

                    enviados += lote.num_rows

                    if (
                        enviados % 100_000 == 0
                        or enviados == total_esperado
                    ):
                        print(
                            f"Enviados: {enviados:,} / "
                            f"{total_esperado:,}",
                            flush=True,
                        )

            if enviados != total_esperado:
                raise RuntimeError("Quantidade enviada divergente.")

            cur.execute(
                sql.SQL("SELECT COUNT(*) FROM {}").format(tabela)
            )
            total_banco = cur.fetchone()[0]

            if total_banco != total_esperado:
                raise RuntimeError("Contagem no banco divergente.")

            for papel in PAPEIS:
                if dict(contagens_arquivo[papel]) != vinculos_esperados[papel]:
                    raise RuntimeError(
                        f"Vínculos do Parquet divergentes: {papel}."
                    )

                coluna_status = sql.Identifier(
                    f"status_vinculo_{papel}"
                )

                cur.execute(
                    sql.SQL(
                        "SELECT {}, COUNT(*) FROM {} GROUP BY {}"
                    ).format(coluna_status, tabela, coluna_status)
                )
                contagens_banco = dict(cur.fetchall())

                if contagens_banco != vinculos_esperados[papel]:
                    raise RuntimeError(
                        f"Vínculos no banco divergentes: {papel}."
                    )

            cur.execute(
                sql.SQL(
                    "SELECT ano_arquivo, COUNT(*) "
                    "FROM {} GROUP BY ano_arquivo"
                ).format(tabela)
            )
            anos_banco = {
                str(ano): quantidade
                for ano, quantidade in cur.fetchall()
            }

            if anos_banco != relatorio["por_ano_arquivo"]:
                raise RuntimeError("Contagens anuais divergentes.")

    print("\nCarga concluída e confirmada!")
    print(f"Registros no banco: {total_banco:,}")
    print("Contagens por ano e vínculos conferidos.")
    print("Ausências e ressalvas preservadas.")


carregar_srag()

Carregando 4,060,608 registros...
Enviados: 100,000 / 4,060,608
Enviados: 200,000 / 4,060,608
Enviados: 300,000 / 4,060,608
Enviados: 400,000 / 4,060,608
Enviados: 500,000 / 4,060,608
Enviados: 600,000 / 4,060,608
Enviados: 700,000 / 4,060,608
Enviados: 800,000 / 4,060,608
Enviados: 900,000 / 4,060,608
Enviados: 1,000,000 / 4,060,608
Enviados: 1,100,000 / 4,060,608
Enviados: 1,200,000 / 4,060,608
Enviados: 1,300,000 / 4,060,608
Enviados: 1,400,000 / 4,060,608
Enviados: 1,500,000 / 4,060,608
Enviados: 1,600,000 / 4,060,608
Enviados: 1,700,000 / 4,060,608
Enviados: 1,800,000 / 4,060,608
Enviados: 1,900,000 / 4,060,608
Enviados: 2,000,000 / 4,060,608
Enviados: 2,100,000 / 4,060,608
Enviados: 2,200,000 / 4,060,608
Enviados: 2,300,000 / 4,060,608
Enviados: 2,400,000 / 4,060,608
Enviados: 2,500,000 / 4,060,608
Enviados: 2,600,000 / 4,060,608
Enviados: 2,700,000 / 4,060,608
Enviados: 2,800,000 / 4,060,608
Enviados: 2,900,000 / 4,060,608
Enviados: 3,000,000 / 4,060,608
Enviados: 3,100,000 / 4,